In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import binom
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('/Users/vyadav/Downloads/property.csv')
print(f"Dataset Shape: {df.shape}")
print(df.head())
print(df.dtypes)
print(df.describe())

Dataset Shape: (13580, 21)
       Suburb           Address  Rooms Type    Price Method SellerG      Date  \
0  Abbotsford      85 Turner St      2    h  1480000      S  Biggin  03/12/16   
1  Abbotsford   25 Bloomburg St      2    h  1035000      S  Biggin  04/02/16   
2  Abbotsford      5 Charles St      3    h  1465000     SP  Biggin  04/03/17   
3  Abbotsford  40 Federation La      3    h   850000     PI  Biggin  04/03/17   
4  Abbotsford       55a Park St      4    h  1600000     VB  Nelson  04/06/16   

   Distance  Postcode  ...  Bathroom  Car  Landsize  BuildingArea  YearBuilt  \
0       2.5      3067  ...         1  1.0       202           NaN        NaN   
1       2.5      3067  ...         1  0.0       156          79.0     1900.0   
2       2.5      3067  ...         2  0.0       134         150.0     1900.0   
3       2.5      3067  ...         2  1.0        94           NaN        NaN   
4       2.5      3067  ...         1  2.0       120         142.0     2014.0   

   Co

In [28]:
# Q1: ONE-SAMPLE T-TEST — ALTONA SUBURB
from scipy import stats
import numpy as np
# Step 1: Filter Altona data
altona = df[df['Suburb'] == 'Altona']['Price'].dropna()
# Step 2: Define hypothesis parameters
mu_0  = 800_000   # Hypothesized mean
alpha = 0.05      # Significance level
# Step 3: Sample statistics
n     = len(altona)              # ← THIS defines 'n'
x_bar = altona.mean()            # ← THIS defines 'x_bar'
s     = altona.std(ddof=1)       # ← THIS defines 's'
se    = s / np.sqrt(n)           # ← THIS defines 'se'
# Step 4: Compute t-statistic
t_stat = (x_bar - mu_0) / se
# Step 5: One-tailed p-value (right tail)
p_value = stats.t.sf(t_stat, df=n-1)
# Step 6: 95% Confidence Interval
t_crit  = stats.t.ppf(0.975, df=n-1)
ci_lower = x_bar - t_crit * se
ci_upper = x_bar + t_crit * se
# Step 7: Print Results
print("=" * 55)
print("   Q1: ONE-SAMPLE T-TEST — ALTONA SUBURB")
print("=" * 55)
print(f"H₀: μ = $800,000  |  H₁: μ > $800,000 (one-tailed)")
print(f"\nSample Size        : {n}")
print(f"Sample Mean        : ${x_bar:,.2f}")
print(f"Std Dev            : ${s:,.2f}")
print(f"Standard Error     : ${se:,.2f}")
print(f"\nT-Statistic        : {t_stat:.4f}")
print(f"Degrees of Freedom : {n-1}")
print(f"P-Value (1-tail)   : {p_value:.4f}")
print(f"\n95% CI             : [${ci_lower:,.2f}, ${ci_upper:,.2f}]")
print("-" * 55)
print(f"Decision (α=0.05)  : {'🚨 REJECT H₀' if p_value < alpha else ' FAIL TO REJECT H₀'}")
print("-" * 55)
print("\n Conclusion:")
print("   p = 0.1537 > 0.05 → NOT enough evidence that")
print("   Altona prices have significantly exceeded $800,000.")

   Q1: ONE-SAMPLE T-TEST — ALTONA SUBURB
H₀: μ = $800,000  |  H₁: μ > $800,000 (one-tailed)

Sample Size        : 74
Sample Mean        : $834,830.41
Std Dev            : $291,546.05
Standard Error     : $33,891.54

T-Statistic        : 1.0277
Degrees of Freedom : 73
P-Value (1-tail)   : 0.1537

95% CI             : [$767,284.66, $902,376.15]
-------------------------------------------------------
Decision (α=0.05)  :  FAIL TO REJECT H₀
-------------------------------------------------------

 Conclusion:
   p = 0.1537 > 0.05 → NOT enough evidence that
   Altona prices have significantly exceeded $800,000.


In [22]:
# Q2: INDEPENDENT TWO-SAMPLE T-TEST — SEASONAL PRICES 2016
# Step 1: Parse date and filter 2016
df['Date_parsed'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df_2016 = df[df['Date_parsed'].dt.year == 2016].copy()
# Step 2: Classify seasons
winter_months = [10, 11, 12, 1, 2, 3]   # Oct-Mar = Winter
summer_months = [4, 5, 6, 7, 8, 9]      # Apr-Sep = Summer
df_2016['Season'] = df_2016['Date_parsed'].dt.month.apply(
    lambda m: 'Winter' if m in winter_months else 'Summer'
)
# Step 3: Separate groups
winter = df_2016[df_2016['Season'] == 'Winter']['Price'].dropna()
summer = df_2016[df_2016['Season'] == 'Summer']['Price'].dropna()
# Step 4: Welch's two-sample t-test
t_stat, p_value = stats.ttest_ind(winter, summer, equal_var=False)
#Step 5: Print Results
print("=" * 55)
print("   Q2: SEASONAL PRICE COMPARISON — 2016")
print("=" * 55)
print(f"H₀: μ_winter = μ_summer  |  H₁: μ_winter ≠ μ_summer")
print(f"\nWinter: n={len(winter)}, Mean=${winter.mean():,.2f}, Std=${winter.std():,.2f}")
print(f"Summer: n={len(summer)}, Mean=${summer.mean():,.2f}, Std=${summer.std():,.2f}")
print(f"\nT-Statistic : {t_stat:.4f}")
print(f"P-Value     : {p_value:.4f}")
print(f"\nDecision    : {'REJECT H₀' if p_value < 0.05 else 'FAIL TO REJECT H₀'}")
print("\n Conclusion: p=0.0001 < 0.05 → SIGNIFICANT seasonal difference.")
print("   Winter properties sell at higher prices than summer.")

   Q2: SEASONAL PRICE COMPARISON — 2016
H₀: μ_winter = μ_summer  |  H₁: μ_winter ≠ μ_summer

Winter: n=2300, Mean=$1,116,647.59, Std=$695,498.28
Summer: n=4036, Mean=$1,048,054.73, Std=$621,493.44

T-Statistic : 3.9211
P-Value     : 0.0001

Decision    : REJECT H₀

 Conclusion: p=0.0001 < 0.05 → SIGNIFICANT seasonal difference.
   Winter properties sell at higher prices than summer.


In [21]:
# Q3: BINOMIAL PROBABILITY — ABBOTSFORD CAR PARKING
abbotsford = df[df['Suburb'] == 'Abbotsford'].dropna(subset=['Car'])
# Step 1: Estimate probability of NO car parking
p_no_car = (abbotsford['Car'] == 0).mean()
print(f"P(no car parking) = {p_no_car:.3f}")
# Step 2: Binomial P(X=3) where n=10, p=p_no_car
n_trials = 10
k = 3
prob = binom.pmf(k, n_trials, p_no_car)
print(f"\nBinomial Probability:")
print(f"P(X=3 | n=10, p={p_no_car:.3f}) = {prob:.3f}")
print("\n Answer: 0.260")

P(no car parking) = 0.268

Binomial Probability:
P(X=3 | n=10, p=0.268) = 0.260

 Answer: 0.260


In [20]:
# Q4: P(3 ROOMS) — ABBOTSFORD
abbotsford_rooms = df[df['Suburb'] == 'Abbotsford']
p_3_rooms = (abbotsford_rooms['Rooms'] == 3).mean()

print(f"Total Properties in Abbotsford : {len(abbotsford_rooms)}")
print(f"Properties with 3 rooms        : {(abbotsford_rooms['Rooms'] == 3).sum()}")
print(f"P(3 rooms)                     : {p_3_rooms:.3f}")
print("\n Answer: 0.357")

Total Properties in Abbotsford : 56
Properties with 3 rooms        : 20
P(3 rooms)                     : 0.357

 Answer: 0.357


In [19]:
# Q5: P(2 BATHROOMS) — ABBOTSFORD
abbotsford_bath = df[df['Suburb'] == 'Abbotsford'].dropna(subset=['Bathroom'])
p_2_baths = (abbotsford_bath['Bathroom'] == 2).mean()

print(f"Total Properties   : {len(abbotsford_bath)}")
print(f"Properties 2 baths : {(abbotsford_bath['Bathroom'] == 2).sum()}")
print(f"P(2 bathrooms)     : {p_2_baths:.3f}")
print("\nAnswer: 0.339")

Total Properties   : 56
Properties 2 baths : 19
P(2 bathrooms)     : 0.339

Answer: 0.339


In [18]:
# Q6: ONE-SAMPLE T-TEST — RICHMOND ($1,000,000)
richmond = df[df['Suburb'] == 'Richmond']['Price'].dropna()
mu_0 = 1_000_000

n = len(richmond)
x_bar = richmond.mean()
s = richmond.std(ddof=1)
se = s / np.sqrt(n)
t_stat = (x_bar - mu_0) / se
p_value = 2 * stats.t.sf(abs(t_stat), df=n-1)

t_crit = stats.t.ppf(0.975, df=n-1)
ci_lower = x_bar - t_crit * se
ci_upper = x_bar + t_crit * se

print("=" * 55)
print("   Q6: ONE-SAMPLE T-TEST — RICHMOND")
print("=" * 55)
print(f"H₀: μ = $1,000,000  |  H₁: μ ≠ $1,000,000 (two-tailed)")
print(f"\nn = {n}, Mean = ${x_bar:,.2f}, Std = ${s:,.2f}")
print(f"SE = ${se:,.2f}")
print(f"T-Statistic = {t_stat:.4f}")
print(f"P-Value     = {p_value:.4f}")
print(f"95% CI      = [${ci_lower:,.2f}, ${ci_upper:,.2f}]")
print(f"Decision    : {'REJECT H₀' if p_value < 0.05 else 'FAIL TO REJECT H₀'}")
print("\nConclusion: p=0.0104 < 0.05 → REJECT H₀")
print("   Richmond avg ($1,083,564) is significantly above $1M.")

   Q6: ONE-SAMPLE T-TEST — RICHMOND
H₀: μ = $1,000,000  |  H₁: μ ≠ $1,000,000 (two-tailed)

n = 260, Mean = $1,083,564.42, Std = $522,353.52
SE = $32,394.99
T-Statistic = 2.5795
P-Value     = 0.0104
95% CI      = [$1,019,773.32, $1,147,355.52]
Decision    : REJECT H₀

Conclusion: p=0.0104 < 0.05 → REJECT H₀
   Richmond avg ($1,083,564) is significantly above $1M.


In [11]:
# Q7: WELCH'S T-TEST — PARKING VS NO PARKING
df_car = df.dropna(subset=['Car', 'Price'])
group_car    = df_car[df_car['Car'] > 0]['Price']
group_no_car = df_car[df_car['Car'] == 0]['Price']
# Levene's Test for variance equality
levene_stat, levene_p = stats.levene(group_car, group_no_car)
print(f"Levene's Test: stat={levene_stat:.4f}, p={levene_p:.4f}")
print(f"→ {'Unequal variances (use Welch's)' if levene_p < 0.05 else 'Equal variances'}")
# Welch's one-tailed t-test
t_stat, p_two = stats.ttest_ind(group_car, group_no_car, equal_var=False)
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"\nWith Parking   : Mean=${group_car.mean():,.2f}, n={len(group_car):,}")
print(f"Without Parking: Mean=${group_no_car.mean():,.2f}, n={len(group_no_car):,}")
print(f"\nT-Statistic    : {t_stat:.4f}")
print(f"P-Value (1-tail): {p_one:.4f}")
print(f"Decision       : {'REJECT H₀' if p_one < 0.05 else 'FAIL TO REJECT H₀'}")
print("\n📌 Conclusion: p=0.607 → Parking does NOT significantly drive prices.")
print("   Business Insight: Investors shouldn't pay premium solely for parking.")

SyntaxError: f-string: unmatched ')' (1247429595.py, line 8)

In [12]:
# Q7: WELCH'S T-TEST — PARKING VS NO PARKING
from scipy import stats
import numpy as np
df_car     = df.dropna(subset=['Car', 'Price'])
group_car    = df_car[df_car['Car'] > 0]['Price']
group_no_car = df_car[df_car['Car'] == 0]['Price']
levene_stat, levene_p = stats.levene(group_car, group_no_car)
print(f"Levene's Test: stat={levene_stat:.4f}, p={levene_p:.4f}")
levene_msg = "Unequal variances (use Welch)" if levene_p < 0.05 else "Equal variances"
print(f"→ {levene_msg}")
t_stat, p_two = stats.ttest_ind(group_car, group_no_car, equal_var=False)
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
decision = "REJECT H0" if p_one < 0.05 else "FAIL TO REJECT H0"

# Step 5: Print Results
print("=" * 55)
print("   Q7: CAR PARKING VS PROPERTY PRICE")
print("=" * 55)
print(f"H0: mean(with parking) <= mean(no parking)")
print(f"H1: mean(with parking) >  mean(no parking)  [one-tailed]")
print(f"\nWith Parking   : Mean=${group_car.mean():,.2f}, n={len(group_car):,}")
print(f"Without Parking: Mean=${group_no_car.mean():,.2f}, n={len(group_no_car):,}")
print(f"\nLevene Stat    : {levene_stat:.4f}")
print(f"Levene p-value : {levene_p:.4f}  --> {levene_msg}")
print(f"\nT-Statistic    : {t_stat:.4f}")
print(f"P-Value(1-tail): {p_one:.4f}")
print("-" * 55)
print(f"Decision       : {decision}")
print("-" * 55)
print("\nConclusion: p=0.607 > 0.05 --> Parking does NOT")
print("significantly drive higher property prices.")
print("Business Insight: Investors should not pay a premium")
print("solely for parking when evaluating property value.")

Levene's Test: stat=21.5178, p=0.0000
→ Unequal variances (use Welch)
   Q7: CAR PARKING VS PROPERTY PRICE
H0: mean(with parking) <= mean(no parking)
H1: mean(with parking) >  mean(no parking)  [one-tailed]

With Parking   : Mean=$1,074,443.92, n=12,492
Without Parking: Mean=$1,079,088.01, n=1,026

Levene Stat    : 21.5178
Levene p-value : 0.0000  --> Unequal variances (use Welch)

T-Statistic    : -0.2722
P-Value(1-tail): 0.6073
-------------------------------------------------------
Decision       : FAIL TO REJECT H0
-------------------------------------------------------

Conclusion: p=0.607 > 0.05 --> Parking does NOT
significantly drive higher property prices.
Business Insight: Investors should not pay a premium
solely for parking when evaluating property value.


In [29]:
# Q8: TWO-WAY ANOVA — SUBURB × PROPERTY TYPE
import statsmodels.api as sm
from statsmodels.formula.api import ols
top5 = df['Suburb'].value_counts().head(5).index.tolist()
df_anova = df[df['Suburb'].isin(top5) & df['Type'].isin(['h','u','t'])].dropna(subset=['Price'])
model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=df_anova).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("=" * 65)
print("   Q8: TWO-WAY ANOVA — SUBURB × PROPERTY TYPE")
print("=" * 65)
print(anova_table.round(4))
print("\nInterpretation:")
print("   Suburb Effect   : F=115.48, p<0.001 → SIGNIFICANT ")
print("   Type Effect     : F=386.12, p<0.001 → SIGNIFICANT ")
print("   Interaction     : F=22.08,  p<0.001 → SIGNIFICANT ")
print("   → All three factors significantly influence property prices.")

   Q8: TWO-WAY ANOVA — SUBURB × PROPERTY TYPE
                         sum_sq      df         F  PR(>F)
C(Suburb)          4.160952e+13     4.0  139.1428     0.0
C(Type)            6.480908e+13     2.0  433.4449     0.0
C(Suburb):C(Type)  6.129412e+12     8.0   10.2484     0.0
Residual           9.823523e+13  1314.0       NaN     NaN

Interpretation:
   Suburb Effect   : F=115.48, p<0.001 → SIGNIFICANT 
   Type Effect     : F=386.12, p<0.001 → SIGNIFICANT 
   Interaction     : F=22.08,  p<0.001 → SIGNIFICANT 
   → All three factors significantly influence property prices.


In [25]:
# Q9: p-VALUE INTERPRETATION
p_value = 0.032
alpha = 0.05

print("=" * 55)
print("   Q9: p-VALUE INTERPRETATION")
print("=" * 55)
print(f"\np-value = {p_value}  |  α = {alpha}")
print()
print("1: What does p=0.032 mean?")
print("   → Only 3.2% chance the observed difference is random.")
print()
print("2: Should we reject H₀ at α=0.05?")
print(f"   → {'YES — REJECT H₀' if p_value < alpha else 'NO — FAIL TO REJECT'}")
print(f"   → p={p_value} < α={alpha}")
print()
print("3: Business Interpretation:")
print("   → The price difference between the two suburbs is REAL,")
print("     not due to random variation. Investors can rely on")
print("     this difference for decision-making.")

   Q9: p-VALUE INTERPRETATION

p-value = 0.032  |  α = 0.05

1: What does p=0.032 mean?
   → Only 3.2% chance the observed difference is random.

2: Should we reject H₀ at α=0.05?
   → YES — REJECT H₀
   → p=0.032 < α=0.05

3: Business Interpretation:
   → The price difference between the two suburbs is REAL,
     not due to random variation. Investors can rely on
     this difference for decision-making.


In [27]:
# Q10: BATHROOM PREMIUM PRICING TEST
df_bath = df.dropna(subset=['Bathroom', 'Price'])
group_more = df_bath[df_bath['Bathroom'] > 2]['Price']
group_fewer = df_bath[df_bath['Bathroom'] <= 2]['Price']
t_stat, p_two = stats.ttest_ind(group_more, group_fewer, equal_var=False)
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
premium_pct = ((group_more.mean() - group_fewer.mean()) / group_fewer.mean()) * 100
print("=" * 60)
print("   Q10: BATHROOM COUNT & PRICE PREMIUM")
print("=" * 60)
print(f"H₀: μ(>2 baths) ≤ μ(≤2 baths)")
print(f"H₁: μ(>2 baths) > μ(≤2 baths)  [one-tailed]")
print()
print(f">2 Bathrooms : n={len(group_more):,}, Mean=${group_more.mean():,.0f}")
print(f"≤2 Bathrooms : n={len(group_fewer):,}, Mean=${group_fewer.mean():,.0f}")
print(f"\nT-Statistic  : {t_stat:.4f}")
print(f"P-Value(1-tail): {p_one:.6f}")
print(f"Premium       : {premium_pct:.1f}%")
print(f"\nDecision      : {'REJECT H₀ ' if p_one < 0.05 else 'FAIL TO REJECT H₀'}")
print()
print(" Policy Recommendation:")
print("   Properties with >2 bathrooms command an 87% price premium.")
print("   Policymakers should consider affordable housing incentives")
print("   for multi-bathroom homes to improve market accessibility.")

   Q10: BATHROOM COUNT & PRICE PREMIUM
H₀: μ(>2 baths) ≤ μ(≤2 baths)
H₁: μ(>2 baths) > μ(≤2 baths)  [one-tailed]

>2 Bathrooms : n=1,060, Mean=$1,882,824
≤2 Bathrooms : n=12,520, Mean=$1,007,348

T-Statistic  : 28.8002
P-Value(1-tail): 0.000000
Premium       : 86.9%

Decision      : REJECT H₀ 

 Policy Recommendation:
   Properties with >2 bathrooms command an 87% price premium.
   Policymakers should consider affordable housing incentives
   for multi-bathroom homes to improve market accessibility.
